# 175. Combine Two Tables

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** database, join
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/combine-two-tables/)

```
Table: Person                     Table: Address
+-------------+---------+         +-------------+---------+
| Column Name | Type    |         | Column Name | Type    |
+-------------+---------+         +-------------+---------+
| personId    | int     |         | addressId   | int     |
| lastName    | varchar |         | personId    | int     |
| firstName   | varchar |         | city        | varchar |
+-------------+---------+         | state       | varchar |
personId is the primary key.      +-------------+---------+
                                  addressId is the primary key.
```

Write a solution to report the first name, last name, city, and state of each person
in the `Person` table. **If the address of a `personId` is not present in the
`Address` table, report `null` instead.**

Return the result table in **any order**.

---

### Example

```
Person:                                Address:
+----------+----------+-----------+    +-----------+----------+---------------+------------+
| personId | lastName | firstName |    | addressId | personId | city          | state      |
+----------+----------+-----------+    +-----------+----------+---------------+------------+
| 1        | Wang     | Allen     |    | 1         | 2        | New York City | New York   |
| 2        | Alice    | Bob       |    | 2         | 3        | Leetcode      | California |
+----------+----------+-----------+    +-----------+----------+---------------+------------+

Output:
+-----------+----------+---------------+----------+
| firstName | lastName | city          | state    |
+-----------+----------+---------------+----------+
| Allen     | Wang     | Null          | Null     |
| Bob       | Alice    | New York City | New York |
+-----------+----------+---------------+----------+
```

Allen Wang has no address, so `city` and `state` come back `null`. Note also that
address `2` belongs to person `3`, who does not exist in `Person` - and does **not**
appear in the output.

---

The first SQL problem, and it is one sentence of real content: *what happens to the
rows that do not match?* Every join you will ever write is a different answer to that
question.

## Before you write anything

**1.** Write the query with a plain `JOIN` first - the one you would type without
thinking - and run it with `show(...)`. Count the rows. The example has two people;
how many come back? Which person vanished, and **why**? Do not read on until you have
run it and seen the row disappear.

**2.** A `JOIN` keeps only the rows that matched on **both** sides. A `LEFT JOIN` keeps
**every** row of the left table, and fills the right-hand columns with `NULL` when
there was no match. Which one does "if the address is not present, report null" ask
for? Say it in one sentence.

**3.** Now the part people get wrong: **which table goes on the left?** Write both

```sql
FROM Person LEFT JOIN Address ON ...
FROM Address LEFT JOIN Person ON ...
```

and run both with `show`. They give different answers. Explain, from the statement,
which one is correct - and say what the wrong one does to person `3`, who has an
address but no `Person` row.

**4.** Both tables have a column called `personId`. What happens if you write
`ON personId = personId`? Try it. Then write the version that is unambiguous, and say
what the rule is. (This is not pedantry - it is the single most common SQL error
message you will ever see.)

**5.** The required output columns are `firstName, lastName, city, state`, in that
order and under those names. Your `SELECT` list controls both. The harness checks the
names as well as the values - say why a grader would care, given that the numbers would
be identical.

**6.** `NULL` is not `0` and it is not `''`. Predict what
`WHERE city = NULL` returns for Allen Wang, then run it. Then run
`WHERE city IS NULL`. That difference will cost you a whole afternoon at some point;
better to pay ten seconds for it now.

## Two routes

**A - `LEFT JOIN`** *(write this first)*

```sql
SELECT p.firstName, p.lastName, a.city, a.state
FROM Person p
LEFT JOIN Address a ON a.personId = p.personId
```

`Person` on the left, so every person survives. Where there is no matching address,
`a.city` and `a.state` are `NULL` - which is exactly what the statement asked for. The
aliases `p` and `a` are what make `ON` unambiguous, and they are worth writing even when
you could get away without them.

**B - a tempting near-miss: a subquery per column**

```sql
SELECT p.firstName, p.lastName,
       (SELECT city  FROM Address a WHERE a.personId = p.personId) AS city,
       (SELECT state FROM Address a WHERE a.personId = p.personId) AS state
FROM Person p
```

Write this one too, then **run the tests on it** - because it fails one, and the failure
is the point. It teaches a genuinely useful fact on the way: a scalar subquery that
finds no rows evaluates to `NULL`, so the addressless people come out right.

But it **fails** the case named *"a person with two addresses gives two rows"*. A scalar
subquery must return one value, so it silently picks one address and drops the other,
where the `LEFT JOIN` correctly returns two rows. The primary key on `Person` means
LeetCode's data never triggers it - which is exactly what makes it worth meeting here,
in a harness that does trigger it.

Write A, run it. Write B, run it, read the failure, and say in one sentence what
assumption B is quietly making about the data.

> **`JOIN` throws non-matching rows away. `LEFT JOIN` keeps them and fills with `NULL`.**
> That is the entire content of this problem, and it is worth over-learning now, because
> from here on the mistake will not delete a row you can see - it will silently delete
> the customer with no orders, the employee with no manager, the day with no sales.

In [3]:
SOLUTION = '''
select 
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [1]:
import sqlite3

SCHEMA = """CREATE TABLE Person (personId INTEGER, lastName TEXT, firstName TEXT);
CREATE TABLE Address (addressId INTEGER, personId INTEGER, city TEXT, state TEXT);"""

EXPECTED_COLUMNS = ['firstName', 'lastName', 'city', 'state']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [4]:
# tests
LEETCODE = '''
INSERT INTO Person VALUES (1,'Wang','Allen'), (2,'Alice','Bob');
INSERT INTO Address VALUES (1,2,'New York City','New York'), (2,3,'Leetcode','California');
'''
check("the LeetCode example", LEETCODE, [
    ('Allen', 'Wang', None, None),
    ('Bob', 'Alice', 'New York City', 'New York'),
])

check("every person has an address", '''
INSERT INTO Person VALUES (1,'A','Ann'), (2,'B','Ben');
INSERT INTO Address VALUES (1,1,'Paris','IDF'), (2,2,'Lyon','ARA');
''', [('Ann','A','Paris','IDF'), ('Ben','B','Lyon','ARA')])

check("question 1: NOBODY has an address - the row must still appear", '''
INSERT INTO Person VALUES (1,'A','Ann'), (2,'B','Ben');
''', [('Ann','A',None,None), ('Ben','B',None,None)])

check("no people at all", '''
INSERT INTO Address VALUES (1,1,'Paris','IDF');
''', [])

check("question 3: an address whose person does not exist is IGNORED", '''
INSERT INTO Person VALUES (1,'A','Ann');
INSERT INTO Address VALUES (1,1,'Paris','IDF'), (2,99,'Ghost','Nowhere');
''', [('Ann','A','Paris','IDF')])

check("a person with two addresses gives two rows", '''
INSERT INTO Person VALUES (1,'A','Ann');
INSERT INTO Address VALUES (1,1,'Paris','IDF'), (2,1,'Nice','PACA');
''', [('Ann','A','Paris','IDF'), ('Ann','A','Nice','PACA')])

check("names that repeat are still separate people", '''
INSERT INTO Person VALUES (1,'Smith','John'), (2,'Smith','John');
INSERT INTO Address VALUES (1,1,'Hull','Yorks');
''', [('John','Smith','Hull','Yorks'), ('John','Smith',None,None)])

check("a mix: some with, some without", '''
INSERT INTO Person VALUES (1,'A','Ann'), (2,'B','Ben'), (3,'C','Cal'), (4,'D','Dot');
INSERT INTO Address VALUES (1,1,'Paris','IDF'), (2,3,'Lyon','ARA');
''', [('Ann','A','Paris','IDF'), ('Ben','B',None,None),
      ('Cal','C','Lyon','ARA'), ('Dot','D',None,None)])

check("both tables empty", '', [])

FAIL the LeetCode example
       SOLUTION is empty - write your query in the cell above
FAIL every person has an address
       SOLUTION is empty - write your query in the cell above
FAIL question 1: NOBODY has an address - the row must still appear
       SOLUTION is empty - write your query in the cell above
FAIL no people at all
       SOLUTION is empty - write your query in the cell above
FAIL question 3: an address whose person does not exist is IGNORED
       SOLUTION is empty - write your query in the cell above
FAIL a person with two addresses gives two rows
       SOLUTION is empty - write your query in the cell above
FAIL names that repeat are still separate people
       SOLUTION is empty - write your query in the cell above
FAIL a mix: some with, some without
       SOLUTION is empty - write your query in the cell above
FAIL both tables empty
       SOLUTION is empty - write your query in the cell above


False

## After it passes

- **Watch the row vanish.** Run your query with `JOIN` instead of `LEFT JOIN` against
  the "NOBODY has an address" dataset using `show(...)`. Zero rows. That is what an
  inner join does to a question about *people*, and seeing it once is worth more than
  reading about it three times.
- **Try `RIGHT JOIN`.** Write `FROM Address a RIGHT JOIN Person p ON ...` and check it
  passes too. Then say why almost nobody writes `RIGHT JOIN` in practice - the answer is
  about reading order, not about capability.
- **Prove question 6 to yourself.** Run `SELECT * FROM Address WHERE city = NULL` and
  then `... WHERE city IS NULL` on a dataset with a missing address. One returns
  nothing, always, even when there *are* nulls. Write down why in one sentence, using
  the words "unknown" and "comparison".
- **Then make it a report.** Add `COALESCE(a.city, 'unknown')` and see the nulls turn
  into a word. That is what a real report does, and it is the moment you decide whether
  "no address on file" and "address is literally unknown" should look the same to a
  reader.
- Siblings: **#183 Customers Who Never Order** (the same `LEFT JOIN`, but you keep the
  *non-matching* rows instead of all of them), #181 Employees Earning More Than Their
  Managers (a join of a table to itself), #1378 Replace Employee ID With The Unique
  Identifier (this problem again, with different nouns).